In [1]:
import pandas as pd
from konlpy.tag import Okt
import numpy as np
import gensim.corpora as corpora
import re
import gensim
from pprint import pprint

In [30]:
df = pd.read_csv('find.csv')
df.head()
len(df)

16708

In [31]:
drop_df = df.drop_duplicates(keep='first')
len(drop_df)

15129

In [32]:
drop_df['href'].fillna(method='ffill',inplace=True)
drop_df['title'].fillna(method='ffill',inplace=True)
drop_df['reviewNum'].fillna(method='ffill',inplace=True)
drop_df['tag'].fillna(method='ffill',inplace=True)
drop_df['brand'].fillna(method='ffill',inplace=True)
drop_df['company'].fillna(method='ffill',inplace=True)
drop_df['howToUse'].fillna(method='ffill',inplace=True)
drop_df['ingredients'].fillna(method='ffill',inplace=True)
drop_df['image'].fillna(method='ffill',inplace=True)
drop_df['volume'].fillna(method='ffill',inplace=True)
drop_df['price'].fillna(method='ffill',inplace=True)
drop_df = drop_df[drop_df['tag'].apply(lambda x: str(x) == '두피샴푸' or str(x) == '탈모샴푸')]
drop_df



/var/folders/6w/c4b_62q127db3fwswnvnsm_c0000gn/T/ipykernel_71139/1727125667.py:1: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  drop_df['href'].fillna(method='ffill',inplace=True)
/var/folders/6w/c4b_62q127db3fwswnvnsm_c0000gn/T/ipykernel_71139/1727125667.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  drop_df['href'].fillna(method='ffill',inplace=True)
/var/folders/6w/c4b_62q127db3fwswnvnsm_c0000gn/T/ipykernel_71139/1727125667.py:2: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  drop_df['title'].fillna(method='ffill',inplace=True)
/var/folders/6w/c4b_62q127db3fwswnvnsm_c0000gn/T/ipykernel_71139/1727125667.py:2: SettingWithC

,href,title,reviewNum,tag,brand,company,howToUse,ingredients,image,volume,price,totalScore,satisfactionScore,priceScore,rebuyScore,commenter,commentDate,commentContent,commentGood,commentBad
0,https://daedamo.com/ingre/27320?sca=탈모관련상품&ove...,\n 블랙포레 두피 쿨&딥클린 탄산쿨링,962.0,탈모샴푸,블랙포레,애경산업,미온수로 모발 및 두피를 충분히 적시고 제품의 적당량을 취하여 두피 및 모발에 가볍...,"정제수,소듐C14-16올레핀설포네이트,소듐메틸코코일타우레이트,코카미도프로필베타인,글...",https://daedamo.com/new/data/file/ingre/210576...,500ml,"41,500원",NaN,80%,80%,100%,K2646350517,3달 전,NaN,"앞머리 모발이식 13년전에 하고는 프로스카 띄엄띄엄 먹었는데, 정수리 광탈 시작...","스댕 용기 개 이쁘고 좋은데, 원가 너무 비싸서 내가 보기에는 가격이 높은 최대 ..."
1,https://daedamo.com/ingre/27320?sca=탈모관련상품&ove...,\n 블랙포레 두피 쿨&딥클린 탄산쿨링,962.0,탈모샴푸,블랙포레,애경산업,미온수로 모발 및 두피를 충분히 적시고 제품의 적당량을 취하여 두피 및 모발에 가볍...,"정제수,소듐C14-16올레핀설포네이트,소듐메틸코코일타우레이트,코카미도프로필베타인,글...",https://daedamo.com/new/data/file/ingre/210576...,500ml,"41,500원",100%,NaN,NaN,NaN,a337*****,한 시간 전,탈모라 써봤는데 시원하고 좋아요,NaN,NaN
2,https://daedamo.com/ingre/27320?sca=탈모관련상품&ove...,\n 블랙포레 두피 쿨&딥클린 탄산쿨링,962.0,탈모샴푸,블랙포레,애경산업,미온수로 모발 및 두피를 충분히 적시고 제품의 적당량을 취하여 두피 및 모발에 가볍...,"정제수,소듐C14-16올레핀설포네이트,소듐메틸코코일타우레이트,코카미도프로필베타인,글...",https://daedamo.com/new/data/file/ingre/210576...,500ml,"41,500원",100%,NaN,NaN,NaN,suwo******,하루 전,머리감을때마다 시원하고좋아요,NaN,NaN
3,https://daedamo.com/ingre/27320?sca=탈모관련상품&ove...,\n 블랙포레 두피 쿨&딥클린 탄산쿨링,962.0,탈모샴푸,블랙포레,애경산업,미온수로 모발 및 두피를 충분히 적시고 제품의 적당량을 취하여 두피 및 모발에 가볍...,"정제수,소듐C14-16올레핀설포네이트,소듐메틸코코일타우레이트,코카미도프로필베타인,글...",https://daedamo.com/new/data/file/ingre/210576...,500ml,"41,500원",100%,NaN,NaN,NaN,wall***,하루 전,전에 쿨샴푸를 한번썻는데 맘에들어서 다른색으로 하나 더 주문했습니다. 샘플 사은품...,NaN,NaN
4,https://daedamo.com/ingre/27320?sca=탈모관련상품&ove...,\n 블랙포레 두피 쿨&딥클린 탄산쿨링,962.0,탈모샴푸,블랙포레,애경산업,미온수로 모발 및 두피를 충분히 적시고 제품의 적당량을 취하여 두피 및 모발에 가볍...,"정제수,소듐C14-16올레핀설포네이트,소듐메틸코코일타우레이트,코카미도프로필베타인,글...",https://daedamo.com/new/data/file/ingre/210576...,500ml,"41,500원",100%,NaN,NaN,NaN,zzzz****,2일 전,아주좋습니다좋아요~,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16689,https://daedamo.com/ingre/124?sca=탈모관련상품&overl...,\n DS래보래토리즈 라디아 샴푸,0.0,두피샴푸,DS래보래토리즈,니옥신,"하루에 한번, 적당량을 머리에 바른 후 마사지를 해줍니다. 1-2분정도 후 다시 마...","정제수,소듐C14-16올레핀설포네이트,코카미도프로필베타인,디 소듐라우레스설포석시네이...",https://daedamo.com/new/data/file/ingre/179434...,180ml,"55,000원",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
16691,https://daedamo.com/ingre/122?sca=탈모관련상품&overl...,\n DS래보래토리즈 니아샴푸,0.0,두피샴푸,DS래보래토리즈,니옥신,"적당량을 두피에 도포하여 1분동안 마사지 하고, 3-5분 후 미지근한 물로 헹구어 ...","정제수, 소듐코코일이세치오네이트,소듐라우릴설포아세테이트, 디소듐라우레스설포석시네이트...",https://daedamo.com/new/data/file/ingre/179434...,180ml,"98,000원",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
16699,https://daedamo.com/ingre/103?sca=탈모관련상품&overl...,\n 팜파스 내츄럴 스켈프 샴푸,0.0,두피샴푸,팜파스,니옥신,미온수로 모발을 충분히 적시고 작당량의 샴푸를 덜어 깨끗하게 세정합니다. 다시 소량...,"정제수, 소듐라우레스설페이트, 암모늄라우릴설페이트, 코카미도프로필베타인, 프로필렌글...",https://daedamo.com/new/data/file/ingre/179434...,550ml+170ml,"29,500원",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
16703,https://daedamo.com/ingre/99?sca=탈모관련상품&overla...,\n 니심 건성모발용 샴푸,0.0,두피샴푸,니심,니옥신,두피와 모발 전체를 마사지 하듯이 1분가량 꼼꼼하게 샴푸하고 미온수로 씻어냅니다. ...,"정제수, 소듐C14-16올레핀설포네이트, 소듐코코암포아세테이트, 피이지-4레이프씨다...",https://daedamo.com/new/data/file/ingre/179434...,240ml,"36,000원",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [33]:
# df = drop_df
# df.nunique()
drop_df['tag'].unique()

array(['탈모샴푸', '두피샴푸'], dtype=object)

In [180]:
drop_df[drop_df['commentContent'].apply(lambda x: str(x).strip()).notnull()]

,href,title,reviewNum,tag,brand,company,howToUse,ingredients,image,volume,...,commentBad,contentTopic,contentTopicPerc,contentTopicDist,goodTopic,goodTopicPerc,goodTopicDist,badTopic,badTopicPerc,badTopicDist
0,https://daedamo.com/ingre/27320?sca=탈모관련상품&ove...,\n 블랙포레 두피 쿨&딥클린 탄산쿨링,962.0,탈모샴푸,블랙포레,애경산업,미온수로 모발 및 두피를 충분히 적시고 제품의 적당량을 취하여 두피 및 모발에 가볍...,"정제수,소듐C14-16올레핀설포네이트,소듐메틸코코일타우레이트,코카미도프로필베타인,글...",https://daedamo.com/new/data/file/ingre/210576...,500ml,...,"스댕 용기 개 이쁘고 좋은데, 원가 너무 비싸서 내가 보기에는 가격이 높은 최대 ...",NaN,NaN,NaN,0.0,0.9935,"[(0, 0.99348485)]",7.0,0.9738,"[(7, 0.97383237)]"
1,https://daedamo.com/ingre/27320?sca=탈모관련상품&ove...,\n 블랙포레 두피 쿨&딥클린 탄산쿨링,962.0,탈모샴푸,블랙포레,애경산업,미온수로 모발 및 두피를 충분히 적시고 제품의 적당량을 취하여 두피 및 모발에 가볍...,"정제수,소듐C14-16올레핀설포네이트,소듐메틸코코일타우레이트,코카미도프로필베타인,글...",https://daedamo.com/new/data/file/ingre/210576...,500ml,...,NaN,0.0,0.8566,"[(0, 0.85661465), (1, 0.023887016), (2, 0.0239...",NaN,NaN,NaN,NaN,NaN,NaN
2,https://daedamo.com/ingre/27320?sca=탈모관련상품&ove...,\n 블랙포레 두피 쿨&딥클린 탄산쿨링,962.0,탈모샴푸,블랙포레,애경산업,미온수로 모발 및 두피를 충분히 적시고 제품의 적당량을 취하여 두피 및 모발에 가볍...,"정제수,소듐C14-16올레핀설포네이트,소듐메틸코코일타우레이트,코카미도프로필베타인,글...",https://daedamo.com/new/data/file/ingre/210576...,500ml,...,NaN,3.0,0.8567,"[(0, 0.02387923), (1, 0.023864381), (2, 0.0238...",NaN,NaN,NaN,NaN,NaN,NaN
3,https://daedamo.com/ingre/27320?sca=탈모관련상품&ove...,\n 블랙포레 두피 쿨&딥클린 탄산쿨링,962.0,탈모샴푸,블랙포레,애경산업,미온수로 모발 및 두피를 충분히 적시고 제품의 적당량을 취하여 두피 및 모발에 가볍...,"정제수,소듐C14-16올레핀설포네이트,소듐메틸코코일타우레이트,코카미도프로필베타인,글...",https://daedamo.com/new/data/file/ingre/210576...,500ml,...,NaN,3.0,0.9548,"[(3, 0.95478183)]",NaN,NaN,NaN,NaN,NaN,NaN
4,https://daedamo.com/ingre/27320?sca=탈모관련상품&ove...,\n 블랙포레 두피 쿨&딥클린 탄산쿨링,962.0,탈모샴푸,블랙포레,애경산업,미온수로 모발 및 두피를 충분히 적시고 제품의 적당량을 취하여 두피 및 모발에 가볍...,"정제수,소듐C14-16올레핀설포네이트,소듐메틸코코일타우레이트,코카미도프로필베타인,글...",https://daedamo.com/new/data/file/ingre/210576...,500ml,...,NaN,2.0,0.7850,"[(0, 0.035856076), (1, 0.035754804), (2, 0.785...",NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16689,https://daedamo.com/ingre/124?sca=탈모관련상품&overl...,\n DS래보래토리즈 라디아 샴푸,0.0,두피샴푸,DS래보래토리즈,니옥신,"하루에 한번, 적당량을 머리에 바른 후 마사지를 해줍니다. 1-2분정도 후 다시 마...","정제수,소듐C14-16올레핀설포네이트,코카미도프로필베타인,디 소듐라우레스설포석시네이...",https://daedamo.com/new/data/file/ingre/179434...,180ml,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
16691,https://daedamo.com/ingre/122?sca=탈모관련상품&overl...,\n DS래보래토리즈 니아샴푸,0.0,두피샴푸,DS래보래토리즈,니옥신,"적당량을 두피에 도포하여 1분동안 마사지 하고, 3-5분 후 미지근한 물로 헹구어 ...","정제수, 소듐코코일이세치오네이트,소듐라우릴설포아세테이트, 디소듐라우레스설포석시네이트...",https://daedamo.com/new/data/file/ingre/179434...,180ml,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
16699,https://daedamo.com/ingre/103?sca=탈모관련상품&overl...,\n 팜파스 내츄럴 스켈프 샴푸,0.0,두피샴푸,팜파스,니옥신,미온수로 모발을 충분히 적시고 작당량의 샴푸를 덜어 깨끗하게 세정합니다. 다시 소량...,"정제수, 소듐라우레스설페이트, 암모늄라우릴설페이트, 코카미도프로필베타인, 프로필렌글...",https://daedamo.com/new/data/file/ingre/179434...,550ml+170ml,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
16703,https://daedamo.com/ingre/99?sca=탈모관련상품&overla...,\n 니심 건성모발용 샴푸,0.0,두피샴푸,니심,니옥신,두피와 모발 전체를 마사지 하듯이 1분가량 꼼꼼하게 샴푸하고 미온수로 씻어냅니다. ...,"정제수, 소듐C14-16올레핀설포네이트, 소듐코코암포아세테이트, 피이지-4레이프씨다...",https://daedamo.com/new/data/file/ingre/179434...,240ml,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [194]:
content = drop_df[drop_df['commentContent'].notnull()][drop_df['commentContent'].apply(lambda x: str(x) != '' and str(x) != ' ')]['commentContent']
good = drop_df[drop_df['commentGood'].notnull()][drop_df['commentGood'].apply(lambda x: str(x) != '' and str(x) != ' ')]['commentGood']
bad = drop_df[drop_df['commentBad'].notnull()][df['commentBad'].apply(lambda x: str(x) != '' and str(x) != ' ')]['commentBad']
print(len(content),len(good),len(bad))
for g in good:
    if g == ' ':
        print(1)

916 9776 5707


/var/folders/6w/c4b_62q127db3fwswnvnsm_c0000gn/T/ipykernel_71139/1070345190.py:1: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  content = drop_df[drop_df['commentContent'].notnull()][drop_df['commentContent'].apply(lambda x: str(x) != '' and str(x) != ' ')]['commentContent']
/var/folders/6w/c4b_62q127db3fwswnvnsm_c0000gn/T/ipykernel_71139/1070345190.py:2: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  good = drop_df[drop_df['commentGood'].notnull()][drop_df['commentGood'].apply(lambda x: str(x) != '' and str(x) != ' ')]['commentGood']
/var/folders/6w/c4b_62q127db3fwswnvnsm_c0000gn/T/ipykernel_71139/1070345190.py:3: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  bad = drop_df[drop_df['commentBad'].notnull()][df['commentBad'].apply(lambda x: str(x) != '' and str(x) != ' ')]['commentBad']


In [195]:
tokenizer = Okt()

In [196]:
pattern = re.compile(r'[\n\t]+')
def make_tokens(doc):
    tokens = []
    if doc and (type(doc)==str or not np.isnan(doc)):
        cleaned_doc = pattern.sub(' ',doc).strip()
        phrase = tokenizer.pos(cleaned_doc,norm=True,stem=True)
        tokens = [word[0] for word in phrase if word[0] and (word[1] in ['Noun','Adjective','Verb','Adverb','VerbPrefix','Suffix'] and (word[0] not in ['하다','있다']))]
    return tokens

In [197]:
cont_tokens = []
for t in content:
    cont_tokens.append(make_tokens(t))


In [198]:
len(cont_tokens)


916

In [199]:
good_tokens = []
for t in good:
    good_tokens.append(make_tokens(t))


In [200]:
len(good_tokens)

9776

In [201]:
bad_tokens = []
for t in bad:
    bad_tokens.append(make_tokens(t))


In [202]:
len(bad_tokens)

5707

In [203]:
cont_id2word = corpora.Dictionary(cont_tokens)
cont_corpus = [cont_id2word.doc2bow(text) for text in cont_tokens]

In [204]:
good_id2word = corpora.Dictionary(good_tokens)
good_corpus = [good_id2word.doc2bow(text) for text in good_tokens]

In [205]:
bad_id2word = corpora.Dictionary(bad_tokens)
bad_corpus = [bad_id2word.doc2bow(text) for text in bad_tokens]

In [206]:
import pyLDAvis.gensim
import pickle
import pyLDAvis
import os

In [182]:
num_topics = 5
lda_model5 = gensim.models.LdaMulticore(corpus=cont_corpus,id2word=cont_id2word,num_topics=num_topics,iterations=500)
doc_lda = lda_model5[cont_corpus]

pyLDAvis.enable_notebook()
vis = pyLDAvis.gensim.prepare(lda_model5,cont_corpus,cont_id2word)
pyLDAvis.display(vis)

In [122]:
num_topics = 6
lda_model6 = gensim.models.LdaMulticore(corpus=cont_corpus,id2word=cont_id2word,num_topics=num_topics,iterations=500)
doc_lda = lda_model6[cont_corpus]

pyLDAvis.enable_notebook()
vis = pyLDAvis.gensim.prepare(lda_model6,cont_corpus,cont_id2word)
pyLDAvis.display(vis)

In [214]:
num_topics = 7
lda_model7 = gensim.models.LdaMulticore(corpus=cont_corpus,id2word=cont_id2word,num_topics=num_topics,iterations=500)
doc_lda = lda_model7[cont_corpus]

pyLDAvis.enable_notebook()
vis = pyLDAvis.gensim.prepare(lda_model7,cont_corpus,cont_id2word)
pyLDAvis.display(vis)

In [207]:
def make_topictable(ldamodel,corpus):
    rows = []
    # topic_table = pd.DataFrame()
    for i, topic_list in enumerate(ldamodel[corpus]):
        doc = topic_list[0] if ldamodel.per_word_topics else topic_list
        doc = sorted(doc, key=lambda x: (x[1]), reverse = True)
        # for j, (topic_num, prop_topic) in enumerate(doc):
        #     if j == 0:
        #         topic_table = pd.concat([topic_table,pd.Series([int(topic_num),round(prop_topic,4),topic_list])],ignore_index=True)
        #     else:
        #         break
        for j, (topic_num, prop_topic) in enumerate(doc):
            if j == 0:
                row = [int(topic_num), round(prop_topic, 4), topic_list]
                rows.append(row)
    
    topic_table = pd.concat([pd.Series(row) for row in rows], axis=1).T
    # topic_table.columns = ['Topic_Num', 'Prop_Topic', 'Topic_List']
    return(topic_table)

In [215]:
cont_lda_model = lda_model7

In [216]:
cont_topictable = make_topictable(cont_lda_model,cont_corpus)
cont_topictable
cont_topictable.reset_index()
cont_topictable.columns = ['문서 번호','가장 비중이 높은 토픽의 비중','각 토픽의 비중']
cont_topictable
# topictable = topictable.reset_index()
# topictable.columns = 

,문서 번호,가장 비중이 높은 토픽의 비중,각 토픽의 비중
0,4,0.8565,"[(0, 0.023886472), (1, 0.023898672), (2, 0.023..."
1,2,0.8566,"[(0, 0.023882678), (1, 0.023896847), (2, 0.856..."
2,4,0.9548,"[(4, 0.9547526)]"
3,3,0.7851,"[(0, 0.035796367), (1, 0.035800457), (2, 0.035..."
4,4,0.914,"[(0, 0.014308274), (1, 0.014320795), (2, 0.014..."
...,...,...,...
911,3,0.7138,"[(0, 0.047684133), (1, 0.04778476), (2, 0.0476..."
912,6,0.9427,"[(6, 0.9426668)]"
913,3,0.9945,"[(3, 0.99445224)]"
914,2,0.9761,"[(2, 0.97610927)]"


In [219]:
num_topics = 8
lda_model8 = gensim.models.LdaMulticore(corpus=cont_corpus,id2word=cont_id2word,num_topics=num_topics,iterations=500)
doc_lda = lda_model8[cont_corpus]

pyLDAvis.enable_notebook()
vis = pyLDAvis.gensim.prepare(lda_model8,cont_corpus,cont_id2word)
pyLDAvis.display(vis)

In [222]:
num_topics = 9
lda_model9 = gensim.models.LdaMulticore(corpus=cont_corpus,id2word=cont_id2word,num_topics=num_topics,iterations=500)
doc_lda = lda_model9[cont_corpus]

pyLDAvis.enable_notebook()
vis = pyLDAvis.gensim.prepare(lda_model9,cont_corpus,cont_id2word)
pyLDAvis.display(vis)

In [106]:
num_topics = 10
lda_model10 = gensim.models.LdaMulticore(corpus=cont_corpus,id2word=cont_id2word,num_topics=num_topics,iterations=500)
doc_lda = lda_model10[cont_corpus]

pyLDAvis.enable_notebook()
vis = pyLDAvis.gensim.prepare(lda_model10,cont_corpus,cont_id2word)
pyLDAvis.display(vis)

In [230]:
num_topics = 5
good_model = gensim.models.LdaMulticore(corpus=good_corpus,id2word=good_id2word,num_topics=num_topics,iterations=500)
doc_lda = good_model[good_corpus]

pyLDAvis.enable_notebook()
vis = pyLDAvis.gensim.prepare(good_model,good_corpus,good_id2word)
pyLDAvis.display(vis)

In [226]:
num_topics = 6
good_model2 = gensim.models.LdaMulticore(corpus=good_corpus,id2word=good_id2word,num_topics=num_topics,iterations=500)
doc_lda = good_model2[good_corpus]

pyLDAvis.enable_notebook()
vis = pyLDAvis.gensim.prepare(good_model2,good_corpus,good_id2word)
pyLDAvis.display(vis)

In [58]:
good_topictable = make_topictable(good_model2,good_corpus)
good_topictable.reset_index()
good_topictable.columns = ['문서 번호','가장 비중이 높은 토픽의 비중','각 토픽의 비중']
good_topictable
# topictable = topictable.reset_index()
# topictable.columns = 

,문서 번호,가장 비중이 높은 토픽의 비중,각 토픽의 비중
0,0,0.9935,"[(0, 0.9934838)]"
1,0,0.9847,"[(0, 0.9846805)]"
2,5,0.9649,"[(5, 0.9648945)]"
3,5,0.9852,"[(5, 0.98521113)]"
4,5,0.8083,"[(3, 0.1791781), (5, 0.8083451)]"
...,...,...,...
9869,0,0.8945,"[(0, 0.8945043), (1, 0.021128822), (2, 0.02110..."
9870,3,0.8797,"[(0, 0.024029048), (1, 0.024054876), (2, 0.024..."
9871,1,0.8318,"[(0, 0.03360546), (1, 0.8317967), (2, 0.033646..."
9872,1,0.832,"[(0, 0.033576574), (1, 0.8320373), (2, 0.03359..."


In [232]:
num_topics = 7
bad_model = gensim.models.LdaMulticore(corpus=bad_corpus,id2word=bad_id2word,num_topics=num_topics,iterations=500)
doc_lda = bad_model[bad_corpus]

pyLDAvis.enable_notebook()
vis = pyLDAvis.gensim.prepare(bad_model,bad_corpus,bad_id2word)
pyLDAvis.display(vis)

In [71]:
num_topics = 9
bad_model2 = gensim.models.LdaMulticore(corpus=bad_corpus,id2word=bad_id2word,num_topics=num_topics,iterations=500)
doc_lda = bad_model2[bad_corpus]

pyLDAvis.enable_notebook()
vis = pyLDAvis.gensim.prepare(bad_model2,bad_corpus,bad_id2word)
pyLDAvis.display(vis)

In [72]:
bad_topictable = make_topictable(bad_model2,bad_corpus)
bad_topictable.reset_index()
bad_topictable.columns = ['문서 번호','가장 비중이 높은 토픽의 비중','각 토픽의 비중']
bad_topictable
# topictable = topictable.reset_index()
# topictable.columns = 

,문서 번호,가장 비중이 높은 토픽의 비중,각 토픽의 비중
0,7,0.9738,"[(7, 0.97383285)]"
1,0,0.9477,"[(0, 0.94766504)]"
2,7,0.9738,"[(7, 0.97383237)]"
3,3,0.9778,"[(3, 0.97776026)]"
4,8,0.7035,"[(0, 0.03707921), (1, 0.037047315), (2, 0.0370..."
...,...,...,...
5734,0,0.8729,"[(0, 0.8729384), (1, 0.015891852), (2, 0.01588..."
5735,0,0.8729,"[(0, 0.87293833), (1, 0.015881611), (2, 0.0158..."
5736,4,0.9012,"[(0, 0.012352571), (1, 0.012357348), (2, 0.012..."
5737,6,0.7776,"[(0, 0.027785704), (1, 0.02779318), (2, 0.0277..."


In [74]:
drop_df

,href,title,reviewNum,tag,brand,company,howToUse,ingredients,image,volume,...,commentGood,commentBad,contentTopicPerc,contentTopicDist,goodTopic,goodTopicPerc,goodTopicDist,badTopic,badTopicPerc,badTopicDist
0,https://daedamo.com/ingre/27320?sca=탈모관련상품&ove...,\n 블랙포레 두피 쿨&딥클린 탄산쿨링,962.0,탈모샴푸,블랙포레,애경산업,미온수로 모발 및 두피를 충분히 적시고 제품의 적당량을 취하여 두피 및 모발에 가볍...,"정제수,소듐C14-16올레핀설포네이트,소듐메틸코코일타우레이트,코카미도프로필베타인,글...",https://daedamo.com/new/data/file/ingre/210576...,500ml,...,"앞머리 모발이식 13년전에 하고는 프로스카 띄엄띄엄 먹었는데, 정수리 광탈 시작...","스댕 용기 개 이쁘고 좋은데, 원가 너무 비싸서 내가 보기에는 가격이 높은 최대 ...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,https://daedamo.com/ingre/27320?sca=탈모관련상품&ove...,\n 블랙포레 두피 쿨&딥클린 탄산쿨링,962.0,탈모샴푸,블랙포레,애경산업,미온수로 모발 및 두피를 충분히 적시고 제품의 적당량을 취하여 두피 및 모발에 가볍...,"정제수,소듐C14-16올레핀설포네이트,소듐메틸코코일타우레이트,코카미도프로필베타인,글...",https://daedamo.com/new/data/file/ingre/210576...,500ml,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,https://daedamo.com/ingre/27320?sca=탈모관련상품&ove...,\n 블랙포레 두피 쿨&딥클린 탄산쿨링,962.0,탈모샴푸,블랙포레,애경산업,미온수로 모발 및 두피를 충분히 적시고 제품의 적당량을 취하여 두피 및 모발에 가볍...,"정제수,소듐C14-16올레핀설포네이트,소듐메틸코코일타우레이트,코카미도프로필베타인,글...",https://daedamo.com/new/data/file/ingre/210576...,500ml,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,https://daedamo.com/ingre/27320?sca=탈모관련상품&ove...,\n 블랙포레 두피 쿨&딥클린 탄산쿨링,962.0,탈모샴푸,블랙포레,애경산업,미온수로 모발 및 두피를 충분히 적시고 제품의 적당량을 취하여 두피 및 모발에 가볍...,"정제수,소듐C14-16올레핀설포네이트,소듐메틸코코일타우레이트,코카미도프로필베타인,글...",https://daedamo.com/new/data/file/ingre/210576...,500ml,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,https://daedamo.com/ingre/27320?sca=탈모관련상품&ove...,\n 블랙포레 두피 쿨&딥클린 탄산쿨링,962.0,탈모샴푸,블랙포레,애경산업,미온수로 모발 및 두피를 충분히 적시고 제품의 적당량을 취하여 두피 및 모발에 가볍...,"정제수,소듐C14-16올레핀설포네이트,소듐메틸코코일타우레이트,코카미도프로필베타인,글...",https://daedamo.com/new/data/file/ingre/210576...,500ml,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16689,https://daedamo.com/ingre/124?sca=탈모관련상품&overl...,\n DS래보래토리즈 라디아 샴푸,0.0,두피샴푸,DS래보래토리즈,니옥신,"하루에 한번, 적당량을 머리에 바른 후 마사지를 해줍니다. 1-2분정도 후 다시 마...","정제수,소듐C14-16올레핀설포네이트,코카미도프로필베타인,디 소듐라우레스설포석시네이...",https://daedamo.com/new/data/file/ingre/179434...,180ml,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
16691,https://daedamo.com/ingre/122?sca=탈모관련상품&overl...,\n DS래보래토리즈 니아샴푸,0.0,두피샴푸,DS래보래토리즈,니옥신,"적당량을 두피에 도포하여 1분동안 마사지 하고, 3-5분 후 미지근한 물로 헹구어 ...","정제수, 소듐코코일이세치오네이트,소듐라우릴설포아세테이트, 디소듐라우레스설포석시네이트...",https://daedamo.com/new/data/file/ingre/179434...,180ml,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
16699,https://daedamo.com/ingre/103?sca=탈모관련상품&overl...,\n 팜파스 내츄럴 스켈프 샴푸,0.0,두피샴푸,팜파스,니옥신,미온수로 모발을 충분히 적시고 작당량의 샴푸를 덜어 깨끗하게 세정합니다. 다시 소량...,"정제수, 소듐라우레스설페이트, 암모늄라우릴설페이트, 코카미도프로필베타인, 프로필렌글...",https://daedamo.com/new/data/file/ingre/179434...,550ml+170ml,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
16703,https://daedamo.com/ingre/99?sca=탈모관련상품&overla...,\n 니심 건성모발용 샴푸,0.0,두피샴푸,니심,니옥신,두피와 모발 전체를 마사지 하듯이 1분가량 꼼꼼하게 샴푸하고 미온수로 씻어냅니다. ...,"정제수, 소듐C14-16올레핀설포네이트, 소듐코코암포아세테이트, 피이지-4레이프씨다...",https://daedamo.com/new/data/file/ingre/179434...,240ml,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [84]:
drop_df['contentTopic'] = np.nan
drop_df['contentTopicPerc'] = np.nan
drop_df['contentTopicDist'] = np.nan
drop_df['goodTopic'] = np.nan
drop_df['goodTopicPerc'] = np.nan
drop_df['goodTopicDist'] = np.nan
drop_df['badTopic'] = np.nan
drop_df['badTopicPerc'] = np.nan
drop_df['badTopicDist'] = np.nan
drop_df.head()

# drop_df = drop_df.drop(columns='contentTopic',axis=1)
# drop_df = drop_df.drop(columns = ['contentTopic','contentTopicDist','contentTopicPerc','goodTopic','goodTopicDist','goodTopicPerc','badTopic','badTopicDist','badTopicPerc'])

,href,title,reviewNum,tag,brand,company,howToUse,ingredients,image,volume,...,commentBad,contentTopic,contentTopicPerc,contentTopicDist,goodTopic,goodTopicPerc,goodTopicDist,badTopic,badTopicPerc,badTopicDist
0,https://daedamo.com/ingre/27320?sca=탈모관련상품&ove...,\n 블랙포레 두피 쿨&딥클린 탄산쿨링,962.0,탈모샴푸,블랙포레,애경산업,미온수로 모발 및 두피를 충분히 적시고 제품의 적당량을 취하여 두피 및 모발에 가볍...,"정제수,소듐C14-16올레핀설포네이트,소듐메틸코코일타우레이트,코카미도프로필베타인,글...",https://daedamo.com/new/data/file/ingre/210576...,500ml,...,"스댕 용기 개 이쁘고 좋은데, 원가 너무 비싸서 내가 보기에는 가격이 높은 최대 ...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,https://daedamo.com/ingre/27320?sca=탈모관련상품&ove...,\n 블랙포레 두피 쿨&딥클린 탄산쿨링,962.0,탈모샴푸,블랙포레,애경산업,미온수로 모발 및 두피를 충분히 적시고 제품의 적당량을 취하여 두피 및 모발에 가볍...,"정제수,소듐C14-16올레핀설포네이트,소듐메틸코코일타우레이트,코카미도프로필베타인,글...",https://daedamo.com/new/data/file/ingre/210576...,500ml,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,https://daedamo.com/ingre/27320?sca=탈모관련상품&ove...,\n 블랙포레 두피 쿨&딥클린 탄산쿨링,962.0,탈모샴푸,블랙포레,애경산업,미온수로 모발 및 두피를 충분히 적시고 제품의 적당량을 취하여 두피 및 모발에 가볍...,"정제수,소듐C14-16올레핀설포네이트,소듐메틸코코일타우레이트,코카미도프로필베타인,글...",https://daedamo.com/new/data/file/ingre/210576...,500ml,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,https://daedamo.com/ingre/27320?sca=탈모관련상품&ove...,\n 블랙포레 두피 쿨&딥클린 탄산쿨링,962.0,탈모샴푸,블랙포레,애경산업,미온수로 모발 및 두피를 충분히 적시고 제품의 적당량을 취하여 두피 및 모발에 가볍...,"정제수,소듐C14-16올레핀설포네이트,소듐메틸코코일타우레이트,코카미도프로필베타인,글...",https://daedamo.com/new/data/file/ingre/210576...,500ml,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,https://daedamo.com/ingre/27320?sca=탈모관련상품&ove...,\n 블랙포레 두피 쿨&딥클린 탄산쿨링,962.0,탈모샴푸,블랙포레,애경산업,미온수로 모발 및 두피를 충분히 적시고 제품의 적당량을 취하여 두피 및 모발에 가볍...,"정제수,소듐C14-16올레핀설포네이트,소듐메틸코코일타우레이트,코카미도프로필베타인,글...",https://daedamo.com/new/data/file/ingre/210576...,500ml,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [75]:
contentTopic = cont_topictable['문서 번호']
contentTopicPerc = cont_topictable['가장 비중이 높은 토픽의 비중']
contentTopicDist = cont_topictable['각 토픽의 비중']

goodTopic = good_topictable['문서 번호']
goodTopicPerc = good_topictable['가장 비중이 높은 토픽의 비중']
goodTopicDist = good_topictable['각 토픽의 비중']

badTopic = bad_topictable['문서 번호']
badTopicPerc = bad_topictable['가장 비중이 높은 토픽의 비중']
badTopicDist = bad_topictable['각 토픽의 비중']

In [76]:
contentTopicDist[0]

[(0, 0.85661465),
 (1, 0.023887016),
 (2, 0.023910817),
 (3, 0.023897346),
 (4, 0.023888288),
 (5, 0.023889806),
 (6, 0.023912055)]

In [77]:
cont_hrefs = drop_df[drop_df['tag'].apply(lambda x: str(x)=='두피샴푸' or str(x)=='탈모샴푸')][drop_df['commentContent'].notnull()]['href'].reset_index()
cont_commenter = drop_df[drop_df['tag'].apply(lambda x: str(x)=='두피샴푸' or str(x)=='탈모샴푸')][drop_df['commentContent'].notnull()]['commenter'].reset_index()
cont_comment = drop_df[drop_df['tag'].apply(lambda x: str(x)=='두피샴푸' or str(x)=='탈모샴푸')][drop_df['commentContent'].notnull()]['commentContent'].reset_index()

good_hrefs = drop_df[drop_df['tag'].apply(lambda x: str(x)=='두피샴푸' or str(x)=='탈모샴푸')][drop_df['commentGood'].notnull()]['href'].reset_index()
good_commenter = drop_df[drop_df['tag'].apply(lambda x: str(x)=='두피샴푸' or str(x)=='탈모샴푸')][drop_df['commentGood'].notnull()]['commenter'].reset_index()
good_comment = drop_df[drop_df['tag'].apply(lambda x: str(x)=='두피샴푸' or str(x)=='탈모샴푸')][drop_df['commentGood'].notnull()]['commentGood'].reset_index()

bad_hrefs = drop_df[drop_df['tag'].apply(lambda x: str(x)=='두피샴푸' or str(x)=='탈모샴푸')][drop_df['commentBad'].notnull()]['href'].reset_index()
bad_commenter = drop_df[drop_df['tag'].apply(lambda x: str(x)=='두피샴푸' or str(x)=='탈모샴푸')][drop_df['commentBad'].notnull()]['commenter'].reset_index()
bad_comment = drop_df[drop_df['tag'].apply(lambda x: str(x)=='두피샴푸' or str(x)=='탈모샴푸')][drop_df['commentBad'].notnull()]['commentBad'].reset_index()

# cont_hrefs = cont_hrefs.reset_index()
cont_hrefs
cont_commenter
# cont_comment

,index,commenter
0,1,a337*****
1,2,suwo******
2,3,wall***
3,4,zzzz****
4,5,sj05****
...,...,...
911,1171,bbho****
912,1172,ggod****
913,1173,love*******
914,1174,the7****


In [78]:
cont_comment.index

RangeIndex(start=0, stop=916, step=1)

In [338]:
drop_df['goodTopic'] = None
drop_df['goodTopicPerc'] = None
drop_df['goodTopicDist'] = None
drop_df['badTopic'] = None
drop_df['badTopicPerc'] = None
drop_df['badTopicDist'] = None


/var/folders/6w/c4b_62q127db3fwswnvnsm_c0000gn/T/ipykernel_39991/655073321.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  drop_df['goodTopic'] = None
/var/folders/6w/c4b_62q127db3fwswnvnsm_c0000gn/T/ipykernel_39991/655073321.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  drop_df['goodTopicPerc'] = None
/var/folders/6w/c4b_62q127db3fwswnvnsm_c0000gn/T/ipykernel_39991/655073321.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexe

In [79]:
drop_df

,href,title,reviewNum,tag,brand,company,howToUse,ingredients,image,volume,...,commentGood,commentBad,contentTopicPerc,contentTopicDist,goodTopic,goodTopicPerc,goodTopicDist,badTopic,badTopicPerc,badTopicDist
0,https://daedamo.com/ingre/27320?sca=탈모관련상품&ove...,\n 블랙포레 두피 쿨&딥클린 탄산쿨링,962.0,탈모샴푸,블랙포레,애경산업,미온수로 모발 및 두피를 충분히 적시고 제품의 적당량을 취하여 두피 및 모발에 가볍...,"정제수,소듐C14-16올레핀설포네이트,소듐메틸코코일타우레이트,코카미도프로필베타인,글...",https://daedamo.com/new/data/file/ingre/210576...,500ml,...,"앞머리 모발이식 13년전에 하고는 프로스카 띄엄띄엄 먹었는데, 정수리 광탈 시작...","스댕 용기 개 이쁘고 좋은데, 원가 너무 비싸서 내가 보기에는 가격이 높은 최대 ...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,https://daedamo.com/ingre/27320?sca=탈모관련상품&ove...,\n 블랙포레 두피 쿨&딥클린 탄산쿨링,962.0,탈모샴푸,블랙포레,애경산업,미온수로 모발 및 두피를 충분히 적시고 제품의 적당량을 취하여 두피 및 모발에 가볍...,"정제수,소듐C14-16올레핀설포네이트,소듐메틸코코일타우레이트,코카미도프로필베타인,글...",https://daedamo.com/new/data/file/ingre/210576...,500ml,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,https://daedamo.com/ingre/27320?sca=탈모관련상품&ove...,\n 블랙포레 두피 쿨&딥클린 탄산쿨링,962.0,탈모샴푸,블랙포레,애경산업,미온수로 모발 및 두피를 충분히 적시고 제품의 적당량을 취하여 두피 및 모발에 가볍...,"정제수,소듐C14-16올레핀설포네이트,소듐메틸코코일타우레이트,코카미도프로필베타인,글...",https://daedamo.com/new/data/file/ingre/210576...,500ml,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,https://daedamo.com/ingre/27320?sca=탈모관련상품&ove...,\n 블랙포레 두피 쿨&딥클린 탄산쿨링,962.0,탈모샴푸,블랙포레,애경산업,미온수로 모발 및 두피를 충분히 적시고 제품의 적당량을 취하여 두피 및 모발에 가볍...,"정제수,소듐C14-16올레핀설포네이트,소듐메틸코코일타우레이트,코카미도프로필베타인,글...",https://daedamo.com/new/data/file/ingre/210576...,500ml,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,https://daedamo.com/ingre/27320?sca=탈모관련상품&ove...,\n 블랙포레 두피 쿨&딥클린 탄산쿨링,962.0,탈모샴푸,블랙포레,애경산업,미온수로 모발 및 두피를 충분히 적시고 제품의 적당량을 취하여 두피 및 모발에 가볍...,"정제수,소듐C14-16올레핀설포네이트,소듐메틸코코일타우레이트,코카미도프로필베타인,글...",https://daedamo.com/new/data/file/ingre/210576...,500ml,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16689,https://daedamo.com/ingre/124?sca=탈모관련상품&overl...,\n DS래보래토리즈 라디아 샴푸,0.0,두피샴푸,DS래보래토리즈,니옥신,"하루에 한번, 적당량을 머리에 바른 후 마사지를 해줍니다. 1-2분정도 후 다시 마...","정제수,소듐C14-16올레핀설포네이트,코카미도프로필베타인,디 소듐라우레스설포석시네이...",https://daedamo.com/new/data/file/ingre/179434...,180ml,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
16691,https://daedamo.com/ingre/122?sca=탈모관련상품&overl...,\n DS래보래토리즈 니아샴푸,0.0,두피샴푸,DS래보래토리즈,니옥신,"적당량을 두피에 도포하여 1분동안 마사지 하고, 3-5분 후 미지근한 물로 헹구어 ...","정제수, 소듐코코일이세치오네이트,소듐라우릴설포아세테이트, 디소듐라우레스설포석시네이트...",https://daedamo.com/new/data/file/ingre/179434...,180ml,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
16699,https://daedamo.com/ingre/103?sca=탈모관련상품&overl...,\n 팜파스 내츄럴 스켈프 샴푸,0.0,두피샴푸,팜파스,니옥신,미온수로 모발을 충분히 적시고 작당량의 샴푸를 덜어 깨끗하게 세정합니다. 다시 소량...,"정제수, 소듐라우레스설페이트, 암모늄라우릴설페이트, 코카미도프로필베타인, 프로필렌글...",https://daedamo.com/new/data/file/ingre/179434...,550ml+170ml,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
16703,https://daedamo.com/ingre/99?sca=탈모관련상품&overla...,\n 니심 건성모발용 샴푸,0.0,두피샴푸,니심,니옥신,두피와 모발 전체를 마사지 하듯이 1분가량 꼼꼼하게 샴푸하고 미온수로 씻어냅니다. ...,"정제수, 소듐C14-16올레핀설포네이트, 소듐코코암포아세테이트, 피이지-4레이프씨다...",https://daedamo.com/new/data/file/ingre/179434...,240ml,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [327]:
cont_hrefs

,index,href
0,1,https://daedamo.com/ingre/27320?sca=탈모관련상품&ove...
1,2,https://daedamo.com/ingre/27320?sca=탈모관련상품&ove...
2,3,https://daedamo.com/ingre/27320?sca=탈모관련상품&ove...
3,4,https://daedamo.com/ingre/27320?sca=탈모관련상품&ove...
4,5,https://daedamo.com/ingre/27320?sca=탈모관련상품&ove...
...,...,...
911,1171,https://daedamo.com/ingre/27319?sca=탈모관련상품&ove...
912,1172,https://daedamo.com/ingre/27319?sca=탈모관련상품&ove...
913,1173,https://daedamo.com/ingre/27319?sca=탈모관련상품&ove...
914,1174,https://daedamo.com/ingre/27319?sca=탈모관련상품&ove...


In [85]:
for i in cont_hrefs.index:
    condition = (drop_df['href'] == cont_hrefs['href'][i]) & (drop_df['commenter'] == cont_commenter['commenter'][i]) & (drop_df['commentContent'] == cont_comment['commentContent'][i])
    drop_df.loc[condition,'contentTopic'] = contentTopic[i]
    drop_df.loc[condition,'contentTopicPerc'] = contentTopicPerc[i]
    drop_df.loc[condition,'contentTopicDist'] = str(contentTopicDist[i])
drop_df.head()

/var/folders/6w/c4b_62q127db3fwswnvnsm_c0000gn/T/ipykernel_71139/2414471744.py:5: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[(0, 0.85661465), (1, 0.023887016), (2, 0.023910817), (3, 0.023897346), (4, 0.023888288), (5, 0.023889806), (6, 0.023912055)]' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  drop_df.loc[condition,'contentTopicDist'] = str(contentTopicDist[i])


,href,title,reviewNum,tag,brand,company,howToUse,ingredients,image,volume,...,commentBad,contentTopic,contentTopicPerc,contentTopicDist,goodTopic,goodTopicPerc,goodTopicDist,badTopic,badTopicPerc,badTopicDist
0,https://daedamo.com/ingre/27320?sca=탈모관련상품&ove...,\n 블랙포레 두피 쿨&딥클린 탄산쿨링,962.0,탈모샴푸,블랙포레,애경산업,미온수로 모발 및 두피를 충분히 적시고 제품의 적당량을 취하여 두피 및 모발에 가볍...,"정제수,소듐C14-16올레핀설포네이트,소듐메틸코코일타우레이트,코카미도프로필베타인,글...",https://daedamo.com/new/data/file/ingre/210576...,500ml,...,"스댕 용기 개 이쁘고 좋은데, 원가 너무 비싸서 내가 보기에는 가격이 높은 최대 ...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,https://daedamo.com/ingre/27320?sca=탈모관련상품&ove...,\n 블랙포레 두피 쿨&딥클린 탄산쿨링,962.0,탈모샴푸,블랙포레,애경산업,미온수로 모발 및 두피를 충분히 적시고 제품의 적당량을 취하여 두피 및 모발에 가볍...,"정제수,소듐C14-16올레핀설포네이트,소듐메틸코코일타우레이트,코카미도프로필베타인,글...",https://daedamo.com/new/data/file/ingre/210576...,500ml,...,NaN,0.0,0.8566,"[(0, 0.85661465), (1, 0.023887016), (2, 0.0239...",NaN,NaN,NaN,NaN,NaN,NaN
2,https://daedamo.com/ingre/27320?sca=탈모관련상품&ove...,\n 블랙포레 두피 쿨&딥클린 탄산쿨링,962.0,탈모샴푸,블랙포레,애경산업,미온수로 모발 및 두피를 충분히 적시고 제품의 적당량을 취하여 두피 및 모발에 가볍...,"정제수,소듐C14-16올레핀설포네이트,소듐메틸코코일타우레이트,코카미도프로필베타인,글...",https://daedamo.com/new/data/file/ingre/210576...,500ml,...,NaN,3.0,0.8567,"[(0, 0.02387923), (1, 0.023864381), (2, 0.0238...",NaN,NaN,NaN,NaN,NaN,NaN
3,https://daedamo.com/ingre/27320?sca=탈모관련상품&ove...,\n 블랙포레 두피 쿨&딥클린 탄산쿨링,962.0,탈모샴푸,블랙포레,애경산업,미온수로 모발 및 두피를 충분히 적시고 제품의 적당량을 취하여 두피 및 모발에 가볍...,"정제수,소듐C14-16올레핀설포네이트,소듐메틸코코일타우레이트,코카미도프로필베타인,글...",https://daedamo.com/new/data/file/ingre/210576...,500ml,...,NaN,3.0,0.9548,"[(3, 0.95478183)]",NaN,NaN,NaN,NaN,NaN,NaN
4,https://daedamo.com/ingre/27320?sca=탈모관련상품&ove...,\n 블랙포레 두피 쿨&딥클린 탄산쿨링,962.0,탈모샴푸,블랙포레,애경산업,미온수로 모발 및 두피를 충분히 적시고 제품의 적당량을 취하여 두피 및 모발에 가볍...,"정제수,소듐C14-16올레핀설포네이트,소듐메틸코코일타우레이트,코카미도프로필베타인,글...",https://daedamo.com/new/data/file/ingre/210576...,500ml,...,NaN,2.0,0.7850,"[(0, 0.035856076), (1, 0.035754804), (2, 0.785...",NaN,NaN,NaN,NaN,NaN,NaN


In [86]:
for i in good_hrefs.index:
    condition = (drop_df['href'] == good_hrefs['href'][i]) & (drop_df['commenter'] == good_commenter['commenter'][i]) & (drop_df['commentGood'] == good_comment['commentGood'][i])
    drop_df.loc[condition,'goodTopic'] = goodTopic[i]
    drop_df.loc[condition,'goodTopicPerc'] = goodTopicPerc[i]
    drop_df.loc[condition,'goodTopicDist'] = str(goodTopicDist[i])
drop_df.head()

/var/folders/6w/c4b_62q127db3fwswnvnsm_c0000gn/T/ipykernel_71139/3611672335.py:5: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[(0, 0.9934838)]' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  drop_df.loc[condition,'goodTopicDist'] = str(goodTopicDist[i])


,href,title,reviewNum,tag,brand,company,howToUse,ingredients,image,volume,...,commentBad,contentTopic,contentTopicPerc,contentTopicDist,goodTopic,goodTopicPerc,goodTopicDist,badTopic,badTopicPerc,badTopicDist
0,https://daedamo.com/ingre/27320?sca=탈모관련상품&ove...,\n 블랙포레 두피 쿨&딥클린 탄산쿨링,962.0,탈모샴푸,블랙포레,애경산업,미온수로 모발 및 두피를 충분히 적시고 제품의 적당량을 취하여 두피 및 모발에 가볍...,"정제수,소듐C14-16올레핀설포네이트,소듐메틸코코일타우레이트,코카미도프로필베타인,글...",https://daedamo.com/new/data/file/ingre/210576...,500ml,...,"스댕 용기 개 이쁘고 좋은데, 원가 너무 비싸서 내가 보기에는 가격이 높은 최대 ...",NaN,NaN,NaN,0.0,0.9935,"[(0, 0.99348485)]",NaN,NaN,NaN
1,https://daedamo.com/ingre/27320?sca=탈모관련상품&ove...,\n 블랙포레 두피 쿨&딥클린 탄산쿨링,962.0,탈모샴푸,블랙포레,애경산업,미온수로 모발 및 두피를 충분히 적시고 제품의 적당량을 취하여 두피 및 모발에 가볍...,"정제수,소듐C14-16올레핀설포네이트,소듐메틸코코일타우레이트,코카미도프로필베타인,글...",https://daedamo.com/new/data/file/ingre/210576...,500ml,...,NaN,0.0,0.8566,"[(0, 0.85661465), (1, 0.023887016), (2, 0.0239...",NaN,NaN,NaN,NaN,NaN,NaN
2,https://daedamo.com/ingre/27320?sca=탈모관련상품&ove...,\n 블랙포레 두피 쿨&딥클린 탄산쿨링,962.0,탈모샴푸,블랙포레,애경산업,미온수로 모발 및 두피를 충분히 적시고 제품의 적당량을 취하여 두피 및 모발에 가볍...,"정제수,소듐C14-16올레핀설포네이트,소듐메틸코코일타우레이트,코카미도프로필베타인,글...",https://daedamo.com/new/data/file/ingre/210576...,500ml,...,NaN,3.0,0.8567,"[(0, 0.02387923), (1, 0.023864381), (2, 0.0238...",NaN,NaN,NaN,NaN,NaN,NaN
3,https://daedamo.com/ingre/27320?sca=탈모관련상품&ove...,\n 블랙포레 두피 쿨&딥클린 탄산쿨링,962.0,탈모샴푸,블랙포레,애경산업,미온수로 모발 및 두피를 충분히 적시고 제품의 적당량을 취하여 두피 및 모발에 가볍...,"정제수,소듐C14-16올레핀설포네이트,소듐메틸코코일타우레이트,코카미도프로필베타인,글...",https://daedamo.com/new/data/file/ingre/210576...,500ml,...,NaN,3.0,0.9548,"[(3, 0.95478183)]",NaN,NaN,NaN,NaN,NaN,NaN
4,https://daedamo.com/ingre/27320?sca=탈모관련상품&ove...,\n 블랙포레 두피 쿨&딥클린 탄산쿨링,962.0,탈모샴푸,블랙포레,애경산업,미온수로 모발 및 두피를 충분히 적시고 제품의 적당량을 취하여 두피 및 모발에 가볍...,"정제수,소듐C14-16올레핀설포네이트,소듐메틸코코일타우레이트,코카미도프로필베타인,글...",https://daedamo.com/new/data/file/ingre/210576...,500ml,...,NaN,2.0,0.7850,"[(0, 0.035856076), (1, 0.035754804), (2, 0.785...",NaN,NaN,NaN,NaN,NaN,NaN


In [88]:
for i in bad_hrefs.index:
    condition = (drop_df['href'] == bad_hrefs['href'][i]) & (drop_df['commenter'] == bad_commenter['commenter'][i]) & (drop_df['commentBad'] == bad_comment['commentBad'][i])
    drop_df.loc[condition,'badTopic'] = badTopic[i]
    drop_df.loc[condition,'badTopicPerc'] = badTopicPerc[i]
    drop_df.loc[condition,'badTopicDist'] = str(badTopicDist[i])
drop_df

,href,title,reviewNum,tag,brand,company,howToUse,ingredients,image,volume,...,commentBad,contentTopic,contentTopicPerc,contentTopicDist,goodTopic,goodTopicPerc,goodTopicDist,badTopic,badTopicPerc,badTopicDist
0,https://daedamo.com/ingre/27320?sca=탈모관련상품&ove...,\n 블랙포레 두피 쿨&딥클린 탄산쿨링,962.0,탈모샴푸,블랙포레,애경산업,미온수로 모발 및 두피를 충분히 적시고 제품의 적당량을 취하여 두피 및 모발에 가볍...,"정제수,소듐C14-16올레핀설포네이트,소듐메틸코코일타우레이트,코카미도프로필베타인,글...",https://daedamo.com/new/data/file/ingre/210576...,500ml,...,"스댕 용기 개 이쁘고 좋은데, 원가 너무 비싸서 내가 보기에는 가격이 높은 최대 ...",NaN,NaN,NaN,0.0,0.9935,"[(0, 0.99348485)]",7.0,0.9738,"[(7, 0.97383237)]"
1,https://daedamo.com/ingre/27320?sca=탈모관련상품&ove...,\n 블랙포레 두피 쿨&딥클린 탄산쿨링,962.0,탈모샴푸,블랙포레,애경산업,미온수로 모발 및 두피를 충분히 적시고 제품의 적당량을 취하여 두피 및 모발에 가볍...,"정제수,소듐C14-16올레핀설포네이트,소듐메틸코코일타우레이트,코카미도프로필베타인,글...",https://daedamo.com/new/data/file/ingre/210576...,500ml,...,NaN,0.0,0.8566,"[(0, 0.85661465), (1, 0.023887016), (2, 0.0239...",NaN,NaN,NaN,NaN,NaN,NaN
2,https://daedamo.com/ingre/27320?sca=탈모관련상품&ove...,\n 블랙포레 두피 쿨&딥클린 탄산쿨링,962.0,탈모샴푸,블랙포레,애경산업,미온수로 모발 및 두피를 충분히 적시고 제품의 적당량을 취하여 두피 및 모발에 가볍...,"정제수,소듐C14-16올레핀설포네이트,소듐메틸코코일타우레이트,코카미도프로필베타인,글...",https://daedamo.com/new/data/file/ingre/210576...,500ml,...,NaN,3.0,0.8567,"[(0, 0.02387923), (1, 0.023864381), (2, 0.0238...",NaN,NaN,NaN,NaN,NaN,NaN
3,https://daedamo.com/ingre/27320?sca=탈모관련상품&ove...,\n 블랙포레 두피 쿨&딥클린 탄산쿨링,962.0,탈모샴푸,블랙포레,애경산업,미온수로 모발 및 두피를 충분히 적시고 제품의 적당량을 취하여 두피 및 모발에 가볍...,"정제수,소듐C14-16올레핀설포네이트,소듐메틸코코일타우레이트,코카미도프로필베타인,글...",https://daedamo.com/new/data/file/ingre/210576...,500ml,...,NaN,3.0,0.9548,"[(3, 0.95478183)]",NaN,NaN,NaN,NaN,NaN,NaN
4,https://daedamo.com/ingre/27320?sca=탈모관련상품&ove...,\n 블랙포레 두피 쿨&딥클린 탄산쿨링,962.0,탈모샴푸,블랙포레,애경산업,미온수로 모발 및 두피를 충분히 적시고 제품의 적당량을 취하여 두피 및 모발에 가볍...,"정제수,소듐C14-16올레핀설포네이트,소듐메틸코코일타우레이트,코카미도프로필베타인,글...",https://daedamo.com/new/data/file/ingre/210576...,500ml,...,NaN,2.0,0.7850,"[(0, 0.035856076), (1, 0.035754804), (2, 0.785...",NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16689,https://daedamo.com/ingre/124?sca=탈모관련상품&overl...,\n DS래보래토리즈 라디아 샴푸,0.0,두피샴푸,DS래보래토리즈,니옥신,"하루에 한번, 적당량을 머리에 바른 후 마사지를 해줍니다. 1-2분정도 후 다시 마...","정제수,소듐C14-16올레핀설포네이트,코카미도프로필베타인,디 소듐라우레스설포석시네이...",https://daedamo.com/new/data/file/ingre/179434...,180ml,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
16691,https://daedamo.com/ingre/122?sca=탈모관련상품&overl...,\n DS래보래토리즈 니아샴푸,0.0,두피샴푸,DS래보래토리즈,니옥신,"적당량을 두피에 도포하여 1분동안 마사지 하고, 3-5분 후 미지근한 물로 헹구어 ...","정제수, 소듐코코일이세치오네이트,소듐라우릴설포아세테이트, 디소듐라우레스설포석시네이트...",https://daedamo.com/new/data/file/ingre/179434...,180ml,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
16699,https://daedamo.com/ingre/103?sca=탈모관련상품&overl...,\n 팜파스 내츄럴 스켈프 샴푸,0.0,두피샴푸,팜파스,니옥신,미온수로 모발을 충분히 적시고 작당량의 샴푸를 덜어 깨끗하게 세정합니다. 다시 소량...,"정제수, 소듐라우레스설페이트, 암모늄라우릴설페이트, 코카미도프로필베타인, 프로필렌글...",https://daedamo.com/new/data/file/ingre/179434...,550ml+170ml,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
16703,https://daedamo.com/ingre/99?sca=탈모관련상품&overla...,\n 니심 건성모발용 샴푸,0.0,두피샴푸,니심,니옥신,두피와 모발 전체를 마사지 하듯이 1분가량 꼼꼼하게 샴푸하고 미온수로 씻어냅니다. ...,"정제수, 소듐C14-16올레핀설포네이트, 소듐코코암포아세테이트, 피이지-4레이프씨다...",https://daedamo.com/new/data/file/ingre/179434...,240ml,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [348]:
drop_df[drop_df['badTopic'].notnull()]

,href,title,reviewNum,tag,brand,company,howToUse,ingredients,image,volume,...,commentBad,contentTopic,contentTopicPerc,contentTopicDist,goodTopic,goodTopicPerc,goodTopicDist,badTopic,badTopicPerc,badTopicDist
0,https://daedamo.com/ingre/27320?sca=탈모관련상품&ove...,\n 블랙포레 두피 쿨&딥클린 탄산쿨링,962.0,탈모샴푸,블랙포레,애경산업,미온수로 모발 및 두피를 충분히 적시고 제품의 적당량을 취하여 두피 및 모발에 가볍...,"정제수,소듐C14-16올레핀설포네이트,소듐메틸코코일타우레이트,코카미도프로필베타인,글...",https://daedamo.com/new/data/file/ingre/210576...,500ml,...,"스댕 용기 개 이쁘고 좋은데, 원가 너무 비싸서 내가 보기에는 가격이 높은 최대 ...",None,None,None,0,0.9935,"[(0, 0.9934807)]",1,0.9738,"[(1, 0.97384083)]"
501,https://daedamo.com/ingre/27320?sca=탈모관련상품&ove...,\n 블랙포레 두피 쿨&딥클린 탄산쿨링,962.0,탈모샴푸,블랙포레,애경산업,미온수로 모발 및 두피를 충분히 적시고 제품의 적당량을 취하여 두피 및 모발에 가볍...,"정제수,소듐C14-16올레핀설포네이트,소듐메틸코코일타우레이트,코카미도프로필베타인,글...",https://daedamo.com/new/data/file/ingre/210576...,500ml,...,"음 꼭찾아서 적으라면,, 가격? 탈모샴푸들 다들 비싸서 근데 뭐~ 다비싸니까요~ ...",None,None,None,4,0.987,"[(4, 0.987028)]",5,0.7119,"[(5, 0.71185887), (7, 0.24233761)]"
542,https://daedamo.com/ingre/27320?sca=탈모관련상품&ove...,\n 블랙포레 두피 쿨&딥클린 탄산쿨링,962.0,탈모샴푸,블랙포레,애경산업,미온수로 모발 및 두피를 충분히 적시고 제품의 적당량을 취하여 두피 및 모발에 가볍...,"정제수,소듐C14-16올레핀설포네이트,소듐메틸코코일타우레이트,코카미도프로필베타인,글...",https://daedamo.com/new/data/file/ingre/210576...,500ml,...,"스댕 용기 개 이쁘고 좋은데, 원가 너무 비싸서 내가 보기에는 가격이 높은 최대 ...",None,None,None,0,0.9935,"[(0, 0.9934807)]",1,0.9738,"[(1, 0.97384083)]"
573,https://daedamo.com/ingre/27320?sca=탈모관련상품&ove...,\n 블랙포레 두피 쿨&딥클린 탄산쿨링,962.0,탈모샴푸,블랙포레,애경산업,미온수로 모발 및 두피를 충분히 적시고 제품의 적당량을 취하여 두피 및 모발에 가볍...,"정제수,소듐C14-16올레핀설포네이트,소듐메틸코코일타우레이트,코카미도프로필베타인,글...",https://daedamo.com/new/data/file/ingre/210576...,500ml,...,가격이 조금 비싸긴 합니다. 그래도 새로운 기술이 적용되어 엄청난 쿨링감을 가지고...,None,None,None,4,0.9859,"[(4, 0.98594284)]",7,0.796,"[(0, 0.18455222), (7, 0.7959838)]"
781,https://daedamo.com/ingre/27320?sca=탈모관련상품&ove...,\n 블랙포레 두피 쿨&딥클린 탄산쿨링,962.0,탈모샴푸,블랙포레,애경산업,미온수로 모발 및 두피를 충분히 적시고 제품의 적당량을 취하여 두피 및 모발에 가볍...,"정제수,소듐C14-16올레핀설포네이트,소듐메틸코코일타우레이트,코카미도프로필베타인,글...",https://daedamo.com/new/data/file/ingre/210576...,500ml,...,특별히 없어요,None,None,None,5,0.9835,"[(5, 0.9834675)]",4,0.7035,"[(0, 0.037062634), (1, 0.037055537), (2, 0.037..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14073,https://daedamo.com/ingre/678?sca=탈모관련상품&overl...,\n 포미포미 맥주 샴푸,1.0,탈모샴푸,포미포미,(주)서울화장품,미온수로 모발 및 두피를 충분히 적신 후 적당량을 모발과 두피에 골고루 도포 후 2...,"정제수, 데실글루코사이드, 디소듐라우레스설포석시네이트, 포타슘코코일글리시네이트, 글...",https://daedamo.com/new/data/file/ingre/179434...,520ml,...,머리가 더빠짐 역시 약이 답이다,None,None,None,3,0.8946,"[(0, 0.021041287), (1, 0.021071786), (2, 0.021...",2,0.8729,"[(0, 0.015879462), (1, 0.015887208), (2, 0.872..."
14074,https://daedamo.com/ingre/2853?sca=탈모관련상품&over...,\n 상모단 샴푸,1.0,두피샴푸,어니스트쥬디,(주)서울화장품,물기가 있는 머리카락에 소량 펌핑하여 샴푸잉.,"인삼수, 흑미추출물, 검정콩추출물, 검은깨추출물, 소듐코코일글루타메이트, 데실글루코...",https://daedamo.com/new/data/file/ingre/373194...,250ml,...,한방샴푸라 그런지 한약재 냄새가 난다,None,None,None,2,0.8795,"[(0, 0.024082024), (1, 0.024048617), (2, 0.879...",0,0.8729,"[(0, 0.87293535), (1, 0.015883282), (2, 0.0158..."
14080,https://daedamo.com/ingre/598?sca=탈모관련상품&overl...,\n 하수오 허벌 에센셜 샴푸,1.0,탈모샴푸,피엘하수오,(주)서울화장품,모발이 젖은 상태에서 적당량을 덜어냅니다. 가볍게 마사지 하듯이 거품을 내어 감습니...,"암모늄라우레스설페이트, 편백수, 암모늄라우릴설페이트, 정제수, 메칠폴리실톡산에멀젼,...",https://daedamo.com/new/data/file/ingre/179434...,750ml,...,구하기 어려운 점? 하지만 도움은 안되는 것 같은 점..?,None,None,None,4,0.831,"[(0, 0.033727035), (1, 0.033743743), (2, 0.033...",4,0.9012,"[(0, 0.012351962), (1, 0.012351288), (2, 0.012..."
14081,https://daedamo.com/ingre/1281?sca=탈모관련상품&over...,\n 헤어캅 네츄럴 샴푸,2.0,두피샴푸,헤어캅,(주)서울화장품,미온수로 두피를 충분히 불려주세요. 샴푸 양은 500원 동전크기만큼으로 거품을 내어...,"어성초, 하수오, 고삼, 녹차, 오미자 등",https://daedamo.com/new/data/file/ingre/179434...,500g,...,아직까진 잘모르겠네요,None,None,None,2,0.8314,"[(0, 0.033679746), (1, 0.03383373), (2, 0.8314...",4,0.7777,"[(0, 0.027782874), (1, 0.027783778), (2, 0.027..."


In [349]:
drop_df.to_csv('final_lda.csv',index=False,encoding='utf-8-sig',escapechar='\\')

In [14]:
df['tag'].unique()
df

,href,title,reviewNum,tag,brand,company,howToUse,ingredients,image,volume,...,commentBad,contentTopic,contentTopicPerc,contentTopicDist,goodTopic,goodTopicPerc,goodTopicDist,badTopic,badTopicPerc,badTopicDist
0,https://daedamo.com/ingre/27320?sca=탈모관련상품&ove...,\n 블랙포레 두피 쿨&딥클린 탄산쿨링,962.0,탈모샴푸,블랙포레,애경산업,미온수로 모발 및 두피를 충분히 적시고 제품의 적당량을 취하여 두피 및 모발에 가볍...,"정제수,소듐C14-16올레핀설포네이트,소듐메틸코코일타우레이트,코카미도프로필베타인,글...",https://daedamo.com/new/data/file/ingre/210576...,500ml,...,"스댕 용기 개 이쁘고 좋은데, 원가 너무 비싸서 내가 보기에는 가격이 높은 최대 ...",NaN,NaN,NaN,0.0,0.9935,"[(0, 0.9934807)]",1.0,0.9738,"[(1, 0.97384083)]"
1,https://daedamo.com/ingre/27320?sca=탈모관련상품&ove...,\n 블랙포레 두피 쿨&딥클린 탄산쿨링,962.0,탈모샴푸,블랙포레,애경산업,미온수로 모발 및 두피를 충분히 적시고 제품의 적당량을 취하여 두피 및 모발에 가볍...,"정제수,소듐C14-16올레핀설포네이트,소듐메틸코코일타우레이트,코카미도프로필베타인,글...",https://daedamo.com/new/data/file/ingre/210576...,500ml,...,NaN,4.0,0.8566,"[(0, 0.023957323), (1, 0.02386534), (2, 0.0239...",NaN,NaN,NaN,NaN,NaN,NaN
2,https://daedamo.com/ingre/27320?sca=탈모관련상품&ove...,\n 블랙포레 두피 쿨&딥클린 탄산쿨링,962.0,탈모샴푸,블랙포레,애경산업,미온수로 모발 및 두피를 충분히 적시고 제품의 적당량을 취하여 두피 및 모발에 가볍...,"정제수,소듐C14-16올레핀설포네이트,소듐메틸코코일타우레이트,코카미도프로필베타인,글...",https://daedamo.com/new/data/file/ingre/210576...,500ml,...,NaN,5.0,0.8567,"[(0, 0.023860294), (1, 0.023860116), (2, 0.023...",NaN,NaN,NaN,NaN,NaN,NaN
3,https://daedamo.com/ingre/27320?sca=탈모관련상품&ove...,\n 블랙포레 두피 쿨&딥클린 탄산쿨링,962.0,탈모샴푸,블랙포레,애경산업,미온수로 모발 및 두피를 충분히 적시고 제품의 적당량을 취하여 두피 및 모발에 가볍...,"정제수,소듐C14-16올레핀설포네이트,소듐메틸코코일타우레이트,코카미도프로필베타인,글...",https://daedamo.com/new/data/file/ingre/210576...,500ml,...,NaN,2.0,0.9547,"[(2, 0.954702)]",NaN,NaN,NaN,NaN,NaN,NaN
4,https://daedamo.com/ingre/27320?sca=탈모관련상품&ove...,\n 블랙포레 두피 쿨&딥클린 탄산쿨링,962.0,탈모샴푸,블랙포레,애경산업,미온수로 모발 및 두피를 충분히 적시고 제품의 적당량을 취하여 두피 및 모발에 가볍...,"정제수,소듐C14-16올레핀설포네이트,소듐메틸코코일타우레이트,코카미도프로필베타인,글...",https://daedamo.com/new/data/file/ingre/210576...,500ml,...,NaN,1.0,0.7851,"[(0, 0.035773713), (1, 0.7850679), (2, 0.03578...",NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15124,https://daedamo.com/ingre/99?sca=탈모관련상품&overla...,\n 니심 건성모발용 샴푸,0.0,두피샴푸,니심,니옥신,두피와 모발 전체를 마사지 하듯이 1분가량 꼼꼼하게 샴푸하고 미온수로 씻어냅니다. ...,"정제수, 소듐C14-16올레핀설포네이트, 소듐코코암포아세테이트, 피이지-4레이프씨다...",https://daedamo.com/new/data/file/ingre/179434...,240ml,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
15125,https://daedamo.com/ingre/82?sca=탈모관련상품&overla...,\n 트리코민 덴시파잉 샴푸,0.0,두피샴푸,트리코민,니옥신,거품을 충분히 내신 후 바로 헹구지 마시고 3-5분 가량 그대로 두어 영양성분이 충...,"정제수, 알로에베라잎즙, 에키네시하추출물, 아이소말트, 완두싹추출물, 하이드롤라이즈...",https://daedamo.com/new/data/file/ingre/179434...,177.4ml,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
15126,https://daedamo.com/ingre/80?sca=탈모관련상품&overla...,\n 드림헤어 순간증모제 전용 미스트,0.0,스타일링,드림헤어,니옥신,증모제를 사용하신후 본 제품을 직접적으로 분사하지 마시고 머리위 하늘에 뿌려주듯 분...,"에탄올, 정제수, 아크릴레이트/옥틸아크릴아마이드코폴리머, 녹차추출물, 곰솔잎추출물,...",https://daedamo.com/new/data/file/ingre/179434...,150ml,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
15127,https://daedamo.com/ingre/75?sca=탈모관련상품&overla...,\n 드림헤어 블랙시크릿(휴대용),0.0,헤어커버,드림헤어,니옥신,증모제를 도포후 두피쪽에 증착될 수 있도록 머리를 쓰다듬듯이 살살 털어줍니다.,"레이온, 폴라아마이드, 비오틴, 실크펩타이드, 카퍼트리펩타이드, 대두레시틴, 헤나추출물",https://daedamo.com/new/data/file/ingre/179434...,7g,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [304]:
drop_df.head()

,href,title,reviewNum,tag,brand,company,howToUse,ingredients,image,volume,...,priceScore,rebuyScore,commenter,commentDate,commentContent,commentGood,commentBad,contentTopic,contentTopicPerc,contentTopicDist
0,https://daedamo.com/ingre/27320?sca=탈모관련상품&ove...,\n 블랙포레 두피 쿨&딥클린 탄산쿨링,962.0,탈모샴푸,블랙포레,애경산업,미온수로 모발 및 두피를 충분히 적시고 제품의 적당량을 취하여 두피 및 모발에 가볍...,"정제수,소듐C14-16올레핀설포네이트,소듐메틸코코일타우레이트,코카미도프로필베타인,글...",https://daedamo.com/new/data/file/ingre/210576...,500ml,...,80%,100%,K2646350517,3달 전,NaN,"앞머리 모발이식 13년전에 하고는 프로스카 띄엄띄엄 먹었는데, 정수리 광탈 시작...","스댕 용기 개 이쁘고 좋은데, 원가 너무 비싸서 내가 보기에는 가격이 높은 최대 ...",None,None,None
1,https://daedamo.com/ingre/27320?sca=탈모관련상품&ove...,\n 블랙포레 두피 쿨&딥클린 탄산쿨링,962.0,탈모샴푸,블랙포레,애경산업,미온수로 모발 및 두피를 충분히 적시고 제품의 적당량을 취하여 두피 및 모발에 가볍...,"정제수,소듐C14-16올레핀설포네이트,소듐메틸코코일타우레이트,코카미도프로필베타인,글...",https://daedamo.com/new/data/file/ingre/210576...,500ml,...,NaN,NaN,a337*****,한 시간 전,탈모라 써봤는데 시원하고 좋아요,NaN,NaN,5,0.8567,None
2,https://daedamo.com/ingre/27320?sca=탈모관련상품&ove...,\n 블랙포레 두피 쿨&딥클린 탄산쿨링,962.0,탈모샴푸,블랙포레,애경산업,미온수로 모발 및 두피를 충분히 적시고 제품의 적당량을 취하여 두피 및 모발에 가볍...,"정제수,소듐C14-16올레핀설포네이트,소듐메틸코코일타우레이트,코카미도프로필베타인,글...",https://daedamo.com/new/data/file/ingre/210576...,500ml,...,NaN,NaN,suwo******,하루 전,머리감을때마다 시원하고좋아요,NaN,NaN,None,None,None
3,https://daedamo.com/ingre/27320?sca=탈모관련상품&ove...,\n 블랙포레 두피 쿨&딥클린 탄산쿨링,962.0,탈모샴푸,블랙포레,애경산업,미온수로 모발 및 두피를 충분히 적시고 제품의 적당량을 취하여 두피 및 모발에 가볍...,"정제수,소듐C14-16올레핀설포네이트,소듐메틸코코일타우레이트,코카미도프로필베타인,글...",https://daedamo.com/new/data/file/ingre/210576...,500ml,...,NaN,NaN,wall***,하루 전,전에 쿨샴푸를 한번썻는데 맘에들어서 다른색으로 하나 더 주문했습니다. 샘플 사은품...,NaN,NaN,None,None,None
4,https://daedamo.com/ingre/27320?sca=탈모관련상품&ove...,\n 블랙포레 두피 쿨&딥클린 탄산쿨링,962.0,탈모샴푸,블랙포레,애경산업,미온수로 모발 및 두피를 충분히 적시고 제품의 적당량을 취하여 두피 및 모발에 가볍...,"정제수,소듐C14-16올레핀설포네이트,소듐메틸코코일타우레이트,코카미도프로필베타인,글...",https://daedamo.com/new/data/file/ingre/210576...,500ml,...,NaN,NaN,zzzz****,2일 전,아주좋습니다좋아요~,NaN,NaN,None,None,None


In [ ]:
good_topictable = make_topictable(good_model2,good_corpus)
# good_topictable
good_topictable.reset_index()
good_topictable.columns = ['문서 번호','가장 비중이 높은 토픽의 비중','각 토픽의 비중']
good_topictable
# topictable = topictable.reset_index()
# topictable.columns = 

In [123]:
df = pd.read_csv('final_lda.csv')


In [3]:
df[df['commentContent'].notnull()].to_csv('commentContent.csv',index=False,encoding='utf-8-sig',escapechar='\\')
df[df['commentGood'].notnull()].to_csv('commentGood.csv',index=False,encoding='utf-8-sig',escapechar='\\')
df[df['commentBad'].notnull()].to_csv('commentBad.csv',index=False,encoding='utf-8-sig',escapechar='\\')


In [160]:
content = df[df['contentTopic'].notnull()]#[df['contentTopic'].apply(lambda x: str(x) != '')]
good = df[df['goodTopic'].notnull()]#[df['commentGood'].apply(lambda x: str(x).strip() != '')]
bad = df[df['badTopic'].notnull()]# [df['commentBad'].apply(lambda x: str(x) != '')]

/var/folders/6w/c4b_62q127db3fwswnvnsm_c0000gn/T/ipykernel_71139/2904366165.py:3: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  bad = df[df['badTopic'].notnull()][df['commentBad'].apply(lambda x: str(x) != '')]


In [171]:
content.sort_values(['contentTopic','contentTopicPerc'],inplace=True)
content[content['commentContent'].apply(lambda x: str(x) == ' ')]

/var/folders/6w/c4b_62q127db3fwswnvnsm_c0000gn/T/ipykernel_71139/2716836841.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  content.sort_values(['contentTopic','contentTopicPerc'],inplace=True)


,href,title,reviewNum,tag,brand,company,howToUse,ingredients,image,volume,...,commentBad,contentTopic,contentTopicPerc,contentTopicDist,goodTopic,goodTopicPerc,goodTopicDist,badTopic,badTopicPerc,badTopicDist


In [176]:
good.sort_values(['goodTopic','goodTopicPerc'],inplace=True)
good['commentGood']# .apply(lambda x: str(x) == ' ')]
# good[good['goodTopic'].apply(lambda x: x == 0)]

/var/folders/6w/c4b_62q127db3fwswnvnsm_c0000gn/T/ipykernel_71139/3051956969.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  good.sort_values(['goodTopic','goodTopicPerc'],inplace=True)


1101                                                     
1116                                                     
1169                                                     
1732                                                     
1792                                                     
                              ...                        
6392     자극은 확실히 적은거 같습니다. 다른 제품에 비해 불필요한성분은 다 뺀듯..\n또...
1706     비슷하거나 조금 더 비싼 가격의 탈모샴푸는 이것저것 써봤는데 효과를 잘 모르겠더라...
2325     TS 프리이엄샴푸 적극추천합니다~~!!!\n올뉴플러스 TS샴푸는 인체적용시험을 통...
2092     그냥 무난무난해서 좋은것같습니다. 딱히 뭔가 부족하다는생각은 안들고 그냥 내가 뭐...
4920     박하향은 늘 그렇듯 감을때 시원한 느낌만 주지, 근본적인 치료는 되지 않는다. 박...
Name: commentGood, Length: 9874, dtype: object

In [173]:
bad.sort_values(['badTopic','badTopicPerc'],inplace=True)
bad[bad['commentBad'].apply(lambda x: str(x) == ' ')]
bad[bad['badTopic'].apply(lambda x: x == 0)]

,href,title,reviewNum,tag,brand,company,howToUse,ingredients,image,volume,...,commentBad,contentTopic,contentTopicPerc,contentTopicDist,goodTopic,goodTopicPerc,goodTopicDist,badTopic,badTopicPerc,badTopicDist
1298,https://daedamo.com/ingre/110?sca=탈모관련상품&overl...,\n 알페신 카페인샴푸 C1,418.0,두피샴푸,알페신,애경산업,젖은 머리에 샴푸를 묻혀 거품을 낸 뒤 카페인 복합 성분이 작용할 수 있도록 두피에...,"정제수, 소듐라우레스설페이트, 카페인, 디소듐라우레스설포석시네이트, 라우레스-2, ...",https://daedamo.com/new/data/file/ingre/179434...,250ml,...,,NaN,NaN,NaN,5.0,0.9474,"[(0, 0.010528759), (1, 0.010528349), (2, 0.010...",0.0,0.1111,"[(0, 0.11111111), (1, 0.11111111), (2, 0.11111..."
1340,https://daedamo.com/ingre/110?sca=탈모관련상품&overl...,\n 알페신 카페인샴푸 C1,418.0,두피샴푸,알페신,애경산업,젖은 머리에 샴푸를 묻혀 거품을 낸 뒤 카페인 복합 성분이 작용할 수 있도록 두피에...,"정제수, 소듐라우레스설페이트, 카페인, 디소듐라우레스설포석시네이트, 라우레스-2, ...",https://daedamo.com/new/data/file/ingre/179434...,250ml,...,,NaN,NaN,NaN,0.0,0.9236,"[(0, 0.9235633), (1, 0.015294977), (2, 0.01531...",0.0,0.1111,"[(0, 0.11111111), (1, 0.11111111), (2, 0.11111..."
1346,https://daedamo.com/ingre/110?sca=탈모관련상품&overl...,\n 알페신 카페인샴푸 C1,418.0,두피샴푸,알페신,애경산업,젖은 머리에 샴푸를 묻혀 거품을 낸 뒤 카페인 복합 성분이 작용할 수 있도록 두피에...,"정제수, 소듐라우레스설페이트, 카페인, 디소듐라우레스설포석시네이트, 라우레스-2, ...",https://daedamo.com/new/data/file/ingre/179434...,250ml,...,,NaN,NaN,NaN,0.0,0.7898,"[(0, 0.78976727), (1, 0.042005807), (2, 0.0421...",0.0,0.1111,"[(0, 0.11111111), (1, 0.11111111), (2, 0.11111..."
1393,https://daedamo.com/ingre/110?sca=탈모관련상품&overl...,\n 알페신 카페인샴푸 C1,418.0,두피샴푸,알페신,애경산업,젖은 머리에 샴푸를 묻혀 거품을 낸 뒤 카페인 복합 성분이 작용할 수 있도록 두피에...,"정제수, 소듐라우레스설페이트, 카페인, 디소듐라우레스설포석시네이트, 라우레스-2, ...",https://daedamo.com/new/data/file/ingre/179434...,250ml,...,,NaN,NaN,NaN,4.0,0.9699,"[(4, 0.96994704)]",0.0,0.1111,"[(0, 0.11111111), (1, 0.11111111), (2, 0.11111..."
1396,https://daedamo.com/ingre/110?sca=탈모관련상품&overl...,\n 알페신 카페인샴푸 C1,418.0,두피샴푸,알페신,애경산업,젖은 머리에 샴푸를 묻혀 거품을 낸 뒤 카페인 복합 성분이 작용할 수 있도록 두피에...,"정제수, 소듐라우레스설페이트, 카페인, 디소듐라우레스설포석시네이트, 라우레스-2, ...",https://daedamo.com/new/data/file/ingre/179434...,250ml,...,,NaN,NaN,NaN,4.0,0.9699,"[(4, 0.96994776)]",0.0,0.1111,"[(0, 0.11111111), (1, 0.11111111), (2, 0.11111..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1055,https://daedamo.com/ingre/110?sca=탈모관련상품&overl...,\n 알페신 카페인샴푸 C1,418.0,두피샴푸,알페신,애경산업,젖은 머리에 샴푸를 묻혀 거품을 낸 뒤 카페인 복합 성분이 작용할 수 있도록 두피에...,"정제수, 소듐라우레스설페이트, 카페인, 디소듐라우레스설포석시네이트, 라우레스-2, ...",https://daedamo.com/new/data/file/ingre/179434...,250ml,...,"2~3년 전만 하더라도 해외직구 등을 통해 저렴하게 구매 할 수 있었으나, 국내 ...",NaN,NaN,NaN,4.0,0.6749,"[(3, 0.3090271), (4, 0.6749295)]",0.0,0.9671,"[(0, 0.9670558)]"
4834,https://daedamo.com/ingre/21446?sca=탈모관련상품&ove...,\n 닥터그루트 마이크로바이옴 스케일링 샴푸,11.0,탈모샴푸,닥터그루트,(주)엘지생활건강,"평소에는 세포가 가장 활성화되는 밤에 두피를 깨끗히 해주시고, 두피는 찬바람으로 모...","정제수, 다잇듐라우레스설포석시네이트, 라우릴하이드록시설테인, 다이소듐라우릴설포석시네...",https://daedamo.com/new/data/file/ingre/203432...,280ml,...,"발림성, 헹굼 등 캡슐같은 잔여물이 상쾌하기는 커녕 찝찝하고 불쾌한 느낌만 남습니...",NaN,NaN,NaN,5.0,0.9772,"[(5, 0.9772266)]",0.0,0.9746,"[(0, 0.97458583)]"
8630,https://daedamo.com/ingre/27315?sca=탈모관련상품&ove...,\n 퍼펙트 볼륨 힐링 샴푸,1.0,탈모샴푸,Mostorng,에필로즈,1. 반시계 방향으로 펌프를 열어 개봉해주세요.\n2. 젖은 모발에 적당량의 거품을...,"편백수, 정제수, 소듐14-16올레핀설포네이트.,코코-베타인, 메틸프로판다이올, 소...",https://daedamo.com/new/data/file/ingre/210576...,300ml/10.14 fl.oz,...,좋은 원료가 사용되어서 그런지 상대적으로 가격이 높다는 느낌을 받았습니다. 하지만...,NaN,NaN,NaN,3.0,0.5374,"[(3, 0.53735554), (5, 0.45983097)]",0.0,0.9760,"[(0, 0.9759545)]"
5961,https://daedamo.com/ingre/507?sca=탈모관련상품&overl...,\n 닥터포헤어 폴리젠 샴푸,826.0,탈모샴푸,닥터포헤어,(주)솔레오코스메틱,1일1회 다음과 같이 사용합니다. 미온수로 모발 및 두피를 충분히 적셔줍니다. 적당...,"정제수, 다이소듐라우레스설포석시네이트, 라우릴하이드록시설테인, 코카마이드메틸엠이에이...",https://daedamo.com/new/data/file/ingre/179434...,500ml,...,워낙에 광고를 많이 해서 구매해봤는데 타제품들에 비해 뛰어난점을 찾기능 어려웠습니...,NaN,NaN,NaN,2.0,0.9728,"[(2, 0.9728061)]",0.0,0.9766,"[(0, 0.9765867)]"


In [93]:
df_final = pd.read_csv('final_lda.csv')
df_final = df_final[df_final['tag'].apply(lambda x: str(x) == '두피샴푸' or str(x) == '탈모샴푸')]
df_final


,href,title,reviewNum,tag,brand,company,howToUse,ingredients,image,volume,...,commentBad,contentTopic,contentTopicPerc,contentTopicDist,goodTopic,goodTopicPerc,goodTopicDist,badTopic,badTopicPerc,badTopicDist
0,https://daedamo.com/ingre/27320?sca=탈모관련상품&ove...,\n 블랙포레 두피 쿨&딥클린 탄산쿨링,962.0,탈모샴푸,블랙포레,애경산업,미온수로 모발 및 두피를 충분히 적시고 제품의 적당량을 취하여 두피 및 모발에 가볍...,"정제수,소듐C14-16올레핀설포네이트,소듐메틸코코일타우레이트,코카미도프로필베타인,글...",https://daedamo.com/new/data/file/ingre/210576...,500ml,...,"스댕 용기 개 이쁘고 좋은데, 원가 너무 비싸서 내가 보기에는 가격이 높은 최대 ...",NaN,NaN,NaN,0.0,0.9935,"[(0, 0.9934807)]",1.0,0.9738,"[(1, 0.97384083)]"
1,https://daedamo.com/ingre/27320?sca=탈모관련상품&ove...,\n 블랙포레 두피 쿨&딥클린 탄산쿨링,962.0,탈모샴푸,블랙포레,애경산업,미온수로 모발 및 두피를 충분히 적시고 제품의 적당량을 취하여 두피 및 모발에 가볍...,"정제수,소듐C14-16올레핀설포네이트,소듐메틸코코일타우레이트,코카미도프로필베타인,글...",https://daedamo.com/new/data/file/ingre/210576...,500ml,...,NaN,4.0,0.8566,"[(0, 0.023957323), (1, 0.02386534), (2, 0.0239...",NaN,NaN,NaN,NaN,NaN,NaN
2,https://daedamo.com/ingre/27320?sca=탈모관련상품&ove...,\n 블랙포레 두피 쿨&딥클린 탄산쿨링,962.0,탈모샴푸,블랙포레,애경산업,미온수로 모발 및 두피를 충분히 적시고 제품의 적당량을 취하여 두피 및 모발에 가볍...,"정제수,소듐C14-16올레핀설포네이트,소듐메틸코코일타우레이트,코카미도프로필베타인,글...",https://daedamo.com/new/data/file/ingre/210576...,500ml,...,NaN,5.0,0.8567,"[(0, 0.023860294), (1, 0.023860116), (2, 0.023...",NaN,NaN,NaN,NaN,NaN,NaN
3,https://daedamo.com/ingre/27320?sca=탈모관련상품&ove...,\n 블랙포레 두피 쿨&딥클린 탄산쿨링,962.0,탈모샴푸,블랙포레,애경산업,미온수로 모발 및 두피를 충분히 적시고 제품의 적당량을 취하여 두피 및 모발에 가볍...,"정제수,소듐C14-16올레핀설포네이트,소듐메틸코코일타우레이트,코카미도프로필베타인,글...",https://daedamo.com/new/data/file/ingre/210576...,500ml,...,NaN,2.0,0.9547,"[(2, 0.954702)]",NaN,NaN,NaN,NaN,NaN,NaN
4,https://daedamo.com/ingre/27320?sca=탈모관련상품&ove...,\n 블랙포레 두피 쿨&딥클린 탄산쿨링,962.0,탈모샴푸,블랙포레,애경산업,미온수로 모발 및 두피를 충분히 적시고 제품의 적당량을 취하여 두피 및 모발에 가볍...,"정제수,소듐C14-16올레핀설포네이트,소듐메틸코코일타우레이트,코카미도프로필베타인,글...",https://daedamo.com/new/data/file/ingre/210576...,500ml,...,NaN,1.0,0.7851,"[(0, 0.035773713), (1, 0.7850679), (2, 0.03578...",NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15110,https://daedamo.com/ingre/124?sca=탈모관련상품&overl...,\n DS래보래토리즈 라디아 샴푸,0.0,두피샴푸,DS래보래토리즈,니옥신,"하루에 한번, 적당량을 머리에 바른 후 마사지를 해줍니다. 1-2분정도 후 다시 마...","정제수,소듐C14-16올레핀설포네이트,코카미도프로필베타인,디 소듐라우레스설포석시네이...",https://daedamo.com/new/data/file/ingre/179434...,180ml,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
15112,https://daedamo.com/ingre/122?sca=탈모관련상품&overl...,\n DS래보래토리즈 니아샴푸,0.0,두피샴푸,DS래보래토리즈,니옥신,"적당량을 두피에 도포하여 1분동안 마사지 하고, 3-5분 후 미지근한 물로 헹구어 ...","정제수, 소듐코코일이세치오네이트,소듐라우릴설포아세테이트, 디소듐라우레스설포석시네이트...",https://daedamo.com/new/data/file/ingre/179434...,180ml,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
15120,https://daedamo.com/ingre/103?sca=탈모관련상품&overl...,\n 팜파스 내츄럴 스켈프 샴푸,0.0,두피샴푸,팜파스,니옥신,미온수로 모발을 충분히 적시고 작당량의 샴푸를 덜어 깨끗하게 세정합니다. 다시 소량...,"정제수, 소듐라우레스설페이트, 암모늄라우릴설페이트, 코카미도프로필베타인, 프로필렌글...",https://daedamo.com/new/data/file/ingre/179434...,550ml+170ml,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
15124,https://daedamo.com/ingre/99?sca=탈모관련상품&overla...,\n 니심 건성모발용 샴푸,0.0,두피샴푸,니심,니옥신,두피와 모발 전체를 마사지 하듯이 1분가량 꼼꼼하게 샴푸하고 미온수로 씻어냅니다. ...,"정제수, 소듐C14-16올레핀설포네이트, 소듐코코암포아세테이트, 피이지-4레이프씨다...",https://daedamo.com/new/data/file/ingre/179434...,240ml,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
